# Usage Demo : 


In [ ]:
import copy
import glob
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from piano_transformer.mgeval import core, utils
from sklearn.model_selection import LeaveOneOut
from tqdm import tqdm

## Absolute measurement: statistic analysis


Assign dataset path

In [ ]:
from piano_transformer.utils.metrics import analyze_dataset_mgeval

analyze_dataset_mgeval(
    "experiments/mistral-62M_remi_maestro_v2/output/generated/", max_samples=100
)

In [ ]:
from piano_transformer.utils.metrics import comparing_pairwise_distances_mgeval

comparing_pairwise_distances_mgeval(
    "experiments/mistral-62M_remi_maestro_v2/output/train/",
    "experiments/mistral-62M_remi_maestro_v2/output/generated/",
    "graphics_62M_subset_debug",
    max_samples=100,
)

In [ ]:
from piano_transformer.utils.metrics import fmd

fmd(
    "experiments/mistral-62M_remi_maestro_v2/output/train_subset",
    "experiments/mistral-162M_remi_maestro_v1/output/generated_subset",
)

In [ ]:
set1 = glob.glob("experiments/mistral-62M_remi_maestro_v2/output/train/*.midi")
print(set1)
print(len(set1))
set1 = set1[:25]

construct empty dictionary to fill in measurement across samples

In [ ]:
num_samples = len(set1)

set_eval_init = {
    "total_used_pitch": np.zeros((num_samples, 1)),
    "total_used_note": np.zeros((num_samples, 1)),
    "total_pitch_class_histogram": np.zeros((num_samples, 12)),
    "pitch_range": np.zeros((num_samples, 1)),
    "avg_pitch_shift": np.zeros((num_samples, 1)),
}

set1_eval = copy.deepcopy(set_eval_init)
metrics_list = list(set1_eval.keys())
kwargs = [{}, {"track_num": 0}, {}, {}, {"track_num": 0}]
for j in range(len(metrics_list)):
    for i in tqdm(range(0, num_samples), desc=f"Evaluating {metrics_list[j]}"):
        feature = core.extract_feature(set1[i])
        set1_eval[metrics_list[j]][i] = getattr(core.metrics(), metrics_list[j])(
            feature, **kwargs[j]
        )

repeat for second dataset

In [ ]:
set2 = glob.glob("experiments/mistral-62M_remi_maestro_v2/output/generated/*.midi")
print(set2)
print(len(set2))
set2 = set2[:25]
num_samples = len(set2)
set2_eval = copy.deepcopy(set_eval_init)

for j in range(len(metrics_list)):
    for i in tqdm(
        range(0, num_samples), desc=f"Evaluating {metrics_list[j]} on generated set"
    ):
        feature = core.extract_feature(set2[i])
        try:
            set2_eval[metrics_list[j]][i] = getattr(core.metrics(), metrics_list[j])(
                feature, **kwargs[j]
            )
        except Exception as e:
            print(f"Error evaluating {metrics_list[j]} for {set2[i]}: {e}")

statistic analysis: absolute measurement

In [ ]:
for i in range(0, len(metrics_list)):
    print(metrics_list[i] + ":")
    print("------------------------")
    print("reference_set")
    print("mean: ", np.mean(set1_eval[metrics_list[i]], axis=0))
    print("std: ", np.std(set1_eval[metrics_list[i]], axis=0))

    print("------------------------")
    print("generated_set")
    print("mean: ", np.mean(set2_eval[metrics_list[i]], axis=0))
    print("std: ", np.std(set2_eval[metrics_list[i]], axis=0))


## Relative measurement: generalizes the result among features with various dimensions


the features are sum- marized to 
- the intra-set distances
- the difference of intra-set and inter-set distances.

exhaustive cross-validation for intra-set distances measurement

In [ ]:
loo = LeaveOneOut()
loo.get_n_splits(np.arange(num_samples))
set1_intra = np.zeros((num_samples, len(metrics_list), num_samples - 1))
set2_intra = np.zeros((num_samples, len(metrics_list), num_samples - 1))
for i in range(len(metrics_list)):
    for train_index, test_index in tqdm(
        loo.split(np.arange(num_samples)),
        desc=f"Computing intra-set distances for {metrics_list[i]}",
    ):
        set1_intra[test_index[0]][i] = utils.c_dist(
            set1_eval[metrics_list[i]][test_index],
            set1_eval[metrics_list[i]][train_index],
        )
        set2_intra[test_index[0]][i] = utils.c_dist(
            set2_eval[metrics_list[i]][test_index],
            set2_eval[metrics_list[i]][train_index],
        )


exhaustive cross-validation for inter-set distances measurement

In [ ]:
loo = LeaveOneOut()
loo.get_n_splits(np.arange(num_samples))
sets_inter = np.zeros((num_samples, len(metrics_list), num_samples))

for i in range(len(metrics_list)):
    for train_index, test_index in tqdm(
        loo.split(np.arange(num_samples)),
        desc=f"Computing inter-set distances for {metrics_list[i]}",
    ):
        sets_inter[test_index[0]][i] = utils.c_dist(
            set1_eval[metrics_list[i]][test_index], set2_eval[metrics_list[i]]
        )

visualization of intra-set and inter-set distances

In [ ]:
plot_set1_intra = np.transpose(set1_intra, (1, 0, 2)).reshape(len(metrics_list), -1)
plot_set2_intra = np.transpose(set2_intra, (1, 0, 2)).reshape(len(metrics_list), -1)
plot_sets_inter = np.transpose(sets_inter, (1, 0, 2)).reshape(len(metrics_list), -1)
for i in range(0, len(metrics_list)):
    sns.kdeplot(plot_set1_intra[i], label="intra_set1")
    sns.kdeplot(plot_sets_inter[i], label="inter")
    sns.kdeplot(plot_set2_intra[i], label="intra_set2")

    plt.title(metrics_list[i])
    plt.xlabel("Euclidean distance")
    plt.legend()
    plt.show()

the difference of intra-set and inter-set distances.

In [ ]:
for i in range(0, len(metrics_list)):
    print(metrics_list[i] + ":")
    print("------------------------")
    print("reference_set")
    print(
        "Kullback–Leibler divergence:",
        utils.kl_dist(plot_set1_intra[i], plot_sets_inter[i]),
    )
    print("Overlap area:", utils.overlap_area(plot_set1_intra[i], plot_sets_inter[i]))

    print("generated_set")
    print(
        "Kullback–Leibler divergence:",
        utils.kl_dist(plot_set2_intra[i], plot_sets_inter[i]),
    )
    print("Overlap area:", utils.overlap_area(plot_set2_intra[i], plot_sets_inter[i]))

In [ ]:
from piano_transformer.utils.metrics import compare, oa, kld
from pathlib import Path

print("pitch:")
print(
    "oa:",
    compare(
        Path("output/prompts"),
        Path("output/generated"),
        oa,
        "tokenizer.json",
        feature="pitch",
    ),
)
print(
    "kld:",
    compare(
        Path("output/prompts"),
        Path("output/generated"),
        kld,
        "tokenizer.json",
        feature="pitch",
    ),
)
print("duration:")
print(
    "oa:",
    compare(
        Path("output/prompts"),
        Path("output/generated"),
        oa,
        "tokenizer.json",
        feature="duration",
    ),
)
print(
    "kld:",
    compare(
        Path("output/prompts"),
        Path("output/generated"),
        kld,
        "tokenizer.json",
        feature="duration",
    ),
)
print("velocity:")
print(
    "oa:",
    compare(
        Path("output/prompts"),
        Path("output/generated"),
        oa,
        "tokenizer.json",
        feature="velocity",
    ),
)
print(
    "kld:",
    compare(
        Path("output/prompts"),
        Path("output/generated"),
        kld,
        "tokenizer.json",
        feature="velocity",
    ),
)

In [ ]:
from piano_transformer.utils.metrics import fmd

fmd(Path("output/prompts"), Path("output/generated"))